# Keybed Seg2 real-frame fine-tuning
Balanced synthetic and real training with source-disjoint validation and held-out evaluation.


In [ ]:
import gc, json, math, shutil, sys, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import cv2, numpy as np, torch
from torch.nn import functional as F
INPUT = Path('/kaggle/input')
SRC = next(INPUT.rglob('segnet2.py')).parent
REAL = next((p.parent for p in INPUT.rglob('splits.json')), None)
if REAL is None: raise FileNotFoundError('real Seg2 dataset must contain splits.json')
SYNTHETIC = next(p.parent for p in INPUT.rglob('corners.json') if p.parent != REAL)
REAL_WORK = Path('/kaggle/working/real-seg2')
def unpack_real_dir(name):
    direct = REAL / name
    if direct.is_dir(): return direct
    archive = next(INPUT.rglob(f'{name}.zip'), None)
    if archive is None: raise FileNotFoundError(f'missing {name}/ or {name}.zip')
    target = REAL_WORK / name
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle: bundle.extractall(target)
    nested = target / name
    return nested if nested.is_dir() else target
FRAME_ROOT, MASK_ROOT = unpack_real_dir('frames'), unpack_real_dir('masks')
PACKAGE, OUT = Path('/kaggle/working/kvt'), Path('/kaggle/working/keybed_seg2.pt')
shutil.rmtree(PACKAGE, ignore_errors=True); shutil.copytree(SRC, PACKAGE); sys.path.insert(0, str(PACKAGE.parent))
from kvt.segnet2 import SEG2_INPUT_SIZE, KeybedSegNet2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH, EPOCHS, MASK_SIZE, RASTER = 48, 25, 144, 576
def mask_from_quad(corners):
    big = np.zeros((RASTER, RASTER), dtype=np.uint8)
    cv2.fillPoly(big, [(np.asarray(corners, dtype=np.float64) * RASTER).astype(np.int32)], 255)
    return cv2.resize(big, (MASK_SIZE, MASK_SIZE), interpolation=cv2.INTER_AREA)
synthetic_corners = json.loads((SYNTHETIC / 'corners.json').read_text())
def load_synthetic(stem):
    image = cv2.imread(str(SYNTHETIC / 'frames' / f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
    if image is None: return None
    return cv2.resize(cv2.cvtColor(image, cv2.COLOR_GRAY2RGB), (SEG2_INPUT_SIZE, SEG2_INPUT_SIZE), interpolation=cv2.INTER_AREA), mask_from_quad(synthetic_corners[stem])
with ThreadPoolExecutor(max_workers=8) as workers: synthetic_pairs = [p for p in workers.map(load_synthetic, sorted(synthetic_corners)) if p is not None]
synthetic_images, synthetic_masks = np.stack([p[0] for p in synthetic_pairs]), np.stack([p[1] for p in synthetic_pairs])
splits, labels = json.loads((REAL / 'splits.json').read_text()), json.loads((REAL / 'labels.json').read_text())
split_names, all_names = {k: set(v) for k, v in splits.items()}, set(labels)
if set().union(*split_names.values()) != all_names or sum(map(len, split_names.values())) != len(all_names): raise ValueError('real splits must be disjoint and cover every label')
if not all(split_names.get(k) for k in ('train', 'validation', 'held_out')): raise ValueError('real splits must include train, validation, held_out')
def load_real(name):
    image, mask = cv2.imread(str(FRAME_ROOT / name), cv2.IMREAD_COLOR), cv2.imread(str(MASK_ROOT / name), cv2.IMREAD_GRAYSCALE)
    if image is None or mask is None: raise FileNotFoundError(f'missing real frame or mask: {name}')
    return cv2.resize(cv2.cvtColor(image, cv2.COLOR_BGR2RGB), (SEG2_INPUT_SIZE, SEG2_INPUT_SIZE), interpolation=cv2.INTER_AREA), cv2.resize(mask, (MASK_SIZE, MASK_SIZE), interpolation=cv2.INTER_AREA)
real_pairs = {}
for split, names in split_names.items():
    with ThreadPoolExecutor(max_workers=8) as workers: real_pairs[split] = list(workers.map(load_real, sorted(names)))
def arrays(pairs): return np.stack([p[0] for p in pairs]), np.stack([p[1] for p in pairs])
real_train_images, real_train_masks = arrays(real_pairs['train'])
real_val_images, real_val_masks = arrays(real_pairs['validation'])
real_held_images, real_held_masks = arrays(real_pairs['held_out'])
rng, order = np.random.default_rng(0), np.random.default_rng(0).permutation(len(synthetic_images))
cut, synthetic_val, synthetic_train = max(1, int(len(order) * .08)), None, None
synthetic_val, synthetic_train = order[:cut], order[cut:]
stores = {'synthetic': (torch.from_numpy(synthetic_images), torch.from_numpy(synthetic_masks)), 'real_train': (torch.from_numpy(real_train_images), torch.from_numpy(real_train_masks)), 'real_val': (torch.from_numpy(real_val_images), torch.from_numpy(real_val_masks)), 'real_held': (torch.from_numpy(real_held_images), torch.from_numpy(real_held_masks))}
MEAN, STD = torch.tensor([.485,.456,.406], device=device).view(1,3,1,1), torch.tensor([.229,.224,.225], device=device).view(1,3,1,1)
def batch(name, index, jitter):
    out = stores[name][0][torch.from_numpy(np.ascontiguousarray(index))].to(device).permute(0,3,1,2).float().div_(255)
    if jitter: out = (out * torch.empty(out.shape[0],1,1,1,device=device).uniform_(.7,1.4) + torch.empty(out.shape[0],1,1,1,device=device).uniform_(-.16,.16)).clamp_(0,1)
    return out.sub_(MEAN).div_(STD)
def truth(name, index): return stores[name][1][torch.from_numpy(np.ascontiguousarray(index))].float().div_(255).unsqueeze(1).to(device)
def evaluate(model, name):
    inter = union = 0.0; model.eval()
    with torch.no_grad():
        for start in range(0, len(stores[name][0]), BATCH):
            pick = np.arange(start, min(start+BATCH, len(stores[name][0]))); pred, actual = torch.sigmoid(model(batch(name,pick,False))) > .5, truth(name,pick) > .5
            inter += float((pred & actual).sum()); union += float((pred | actual).sum())
    return inter / max(union, 1.0)
torch.manual_seed(0); model = KeybedSegNet2().to(device); optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
half, step_count = BATCH // 2, max(math.ceil(len(synthetic_train)/ (BATCH//2)), math.ceil(len(real_train_images)/(BATCH//2)))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=step_count*EPOCHS, eta_min=1e-6)
for epoch in range(EPOCHS):
    model.train()
    for _ in range(step_count):
        si, ri = rng.choice(synthetic_train, half, replace=True), rng.choice(len(real_train_images), half, replace=True)
        optimizer.zero_grad(set_to_none=True); loss = F.binary_cross_entropy_with_logits(model(torch.cat([batch('synthetic',si,True), batch('real_train',ri,True)])), torch.cat([truth('synthetic',si), truth('real_train',ri)])); loss.backward(); optimizer.step(); scheduler.step()
    synthetic_iou, real_iou, held_iou = evaluate(model,'synthetic'), evaluate(model,'real_val'), evaluate(model,'real_held')
    torch.save(model.state_dict(), OUT.with_name(f'seg2-e{epoch+1:02d}.pt')); gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print(f'epoch {epoch+1}/{EPOCHS} synthetic_iou {synthetic_iou:.4f} real_val_iou {real_iou:.4f} held_out_iou {held_iou:.4f}', flush=True)
print(f'saved checkpoints to {OUT.parent}')